# Bridge 01 — Indexing, Shape, Axis Operations

These are the exact mechanics that block dojo notebooks 05–07.  
Each problem is 3–8 lines. No hints. Work until the assert passes.

If you are stuck for more than 10 minutes on a single cell: write the primitive version (loops, explicit math), get the assert green, then refactor to the idiomatic version.

In [1]:
import numpy as np
np.random.seed(0)

---
## Section A — Slicing & Indexing

### A1
Given `x` of shape `(10,)`, produce an array of shape `(10,)` where every element at an odd index is negated. Even indices unchanged.

In [9]:
x = np.arange(10, dtype=float)  # [0,1,2,...,9]

out = x.copy()
# YOUR CODE
# out= [-1*x[i] if i%2!=0 else x[i] for i in range(0,len(x))]
out[1::2]*=-1
assert np.allclose(out, [0,-1,2,-3,4,-5,6,-7,8,-9]), out

### A2
Given `A` of shape `(6, 6)`, extract the **anti-diagonal** — elements `A[0,5], A[1,4], A[2,3], A[3,2], A[4,1], A[5,0]` — as a 1D array. No loops.

In [34]:
A = np.arange(36).reshape(6, 6)
# print(A[::])

anti_diag = A[:,::-1].diagonal()  # YOUR CODE

assert anti_diag.shape == (6,)
assert np.allclose(anti_diag, [5, 10, 15, 20, 25, 30]), anti_diag

### A3
Given `X` of shape `(N, D)` and a 1D index array `idx` of shape `(N,)` with values in `[0, D)`, extract `X[i, idx[i]]` for every row — result shape `(N,)`. No loops.

In [43]:
X   = np.array([[1,2,3],[4,5,6],[7,8,9],[10,11,12]], dtype=float)
idx = np.array([2, 0, 1, 2])



gathered = X[np.arange(len(idx)), idx]
print(gathered)

assert gathered.shape == (4,)
assert np.allclose(gathered, [3, 4, 8, 12]), gathered

[ 3.  4.  8. 12.]


### A4 — circular index without `np.roll`
Shift array `x` to the right by `k=3` positions **using only indexing**. Element at position `i` in the output came from position `(i - k) % N` of the input.

In [53]:
x = np.array([10, 20, 30, 40, 50, 60, 70, 80])
k = 3
N = len(x)


last_k = x[-k:]
print(last_k,x[:N-k] )

shifted =  x.copy()# YOUR CODE — use only indexing, not np.roll
shifted[k:]= x[:-k]
shifted[:k] = last_k

assert np.allclose(shifted, np.roll(x, k)), shifted

[60 70 80] [10 20 30 40 50]


### A5 — scatter write with `np.add.at`
Given indices `idx` (may contain repeats) and values `vals`, add each value to position `idx[i]` of a zero array of length 8. The result at position `j` is the sum of all `vals[i]` where `idx[i] == j`.

In [73]:
idx  = np.array([0, 2, 2, 3, 0, 7])
vals = np.array([1, 2, 3, 4, 5, 6], dtype=float)

out = np.zeros(8)
# YOUR CODE — np.add.at(out, idx, vals)
for i in range(len(idx)):
    out[idx[i]] += vals[i]

assert np.allclose(out, [6, 0, 5, 4, 0, 0, 0, 6]), out

### A6 — scatter max with `np.maximum.at`
Same setup but instead of sum, store the **maximum** value seen at each index position.

In [76]:
idx  = np.array([0, 2, 2, 3, 0, 7])
vals = np.array([1, 2, 3, 4, 5, 6], dtype=float)

out = np.zeros(8)  # initial 0 — positions with no value stay 0
# YOUR CODE — np.maximum.at(out, idx, vals)
np.maximum.at(out, idx, vals)

assert np.allclose(out, [5, 0, 3, 4, 0, 0, 0, 6]), out

---
## Section B — Shape & Axis

### B1
Given `X` of shape `(N, H, W)`, flatten only the spatial dims to get `(N, H*W)`. Do it in one operation, no reshape with explicit numbers.

In [54]:
X = np.random.randn(16, 8, 8)

flat = X.reshape(X.shape[0], X.shape[1]* X.shape[2])  # YOUR CODE

assert flat.shape == (16, 64), flat.shape

### B2
Given `A` of shape `(4, 3)` and `B` of shape `(4, 5)`, concatenate along axis=1 to get `(4, 8)`. Then stack two copies of `A` along a **new** axis 0 to get `(2, 4, 3)`.

In [102]:
A = np.ones((4, 3))
B = np.zeros((4, 5))
# print(A, B)
cat   = np.concatenate((A, B), axis =1)  # YOUR CODE
# print(cat.shape)
stack = np.stack((A, A), axis=0)  # YOUR CODE
# print(stack)
assert cat.shape   == (4, 8),    cat.shape
assert stack.shape == (2, 4, 3), stack.shape

### B3
Given `X` of shape `(B, C, H, W)`, move the channel axis to last position to get `(B, H, W, C)`. Do it with a single operation.

In [105]:
X = np.random.randn(8, 3, 32, 32)

out = np.transpose(X , (0, 2, 3, 1))  # YOUR CODE

assert out.shape == (8, 32, 32, 3), out.shape
# Values must be the same, not a copy artifact
assert np.allclose(X[0, :, 5, 7], out[0, 5, 7, :]), 'values changed'

### B4
Given `X` of shape `(N, D)`, compute the **row-wise L2 norm** — shape `(N,)`. Then normalize each row to unit length — shape `(N, D)`. Guard against zero-norm rows.

In [ ]:
X = np.random.randn(10, 5)
X[3] = 0.0  # zero-norm row — should stay zero after normalization

norms      = None  # YOUR CODE — shape (10,)
normalized = None  # YOUR CODE — shape (10, 5)

assert norms.shape == (10,)
assert normalized.shape == (10, 5)
row_norms_after = np.linalg.norm(normalized, axis=1)
assert np.allclose(row_norms_after[np.arange(10) != 3], 1.0, atol=1e-6), row_norms_after
assert np.allclose(normalized[3], 0.0), 'zero row should stay zero'

### B5
Given `X` of shape `(N, D)`, subtract the **column mean** (broadcast correctly) and divide by **column std**. Do both in one line each. Use `keepdims`.

In [ ]:
X = np.random.randn(100, 8) * 3 + 5  # non-zero mean, non-unit std

X_norm = None  # YOUR CODE — subtract col mean, divide by col std

assert X_norm.shape == (100, 8)
assert np.allclose(X_norm.mean(axis=0), 0, atol=1e-6)
assert np.allclose(X_norm.std(axis=0),  1, atol=1e-6)

---
## Section C — Broadcasting

### C1
Given `A` of shape `(N, 1)` and `B` of shape `(1, M)`, produce the **outer difference** matrix of shape `(N, M)` where `out[i, j] = A[i] - B[j]`. No loops, no explicit reshape beyond what's given.

In [ ]:
A = np.array([[1],[3],[5],[7]], dtype=float)  # (4, 1)
B = np.array([[10, 20, 30]], dtype=float)     # (1, 3)

diff = None  # YOUR CODE

assert diff.shape == (4, 3)
assert np.allclose(diff, [[-9,-19,-29],[-7,-17,-27],[-5,-15,-25],[-3,-13,-23]]), diff

### C2
Given `X` of shape `(N, D)` (row vectors) and `Y` of shape `(M, D)` (row vectors), compute the pairwise **squared** L2 distance matrix of shape `(N, M)` using broadcasting. No loops.

Identity: `||x - y||^2 = ||x||^2 + ||y||^2 - 2 x·y^T`

In [ ]:
X = np.array([[1,0],[0,1],[1,1]], dtype=float)  # (3, 2)
Y = np.array([[0,0],[2,2]], dtype=float)         # (2, 2)

sq_dist = None  # YOUR CODE — shape (3, 2)

assert sq_dist.shape == (3, 2)
# Manual check: dist([1,0], [0,0])^2 = 1, dist([1,0], [2,2])^2 = 1+4=5
assert np.allclose(sq_dist[0], [1, 5]), sq_dist[0]
assert np.allclose(sq_dist[2], [2, 2]), sq_dist[2]  # [1,1] to [0,0]=2, [2,2]=2

### C3
Given `boxes` of shape `(N, 4)` with format `[x1, y1, x2, y2]`, compute the area of each box. Then broadcast to get an `(N, N)` matrix of **area products** `area[i] * area[j]`. No loops.

In [ ]:
boxes = np.array([[0,0,4,4],[1,1,3,5],[0,0,2,6]], dtype=float)

areas        = None  # YOUR CODE — shape (3,)
area_product = None  # YOUR CODE — shape (3, 3)

assert areas.shape == (3,)
assert np.allclose(areas, [16, 8, 12]), areas
assert area_product.shape == (3, 3)
assert np.allclose(area_product[0], [256, 128, 192]), area_product[0]

### C4 — the `(N, 1, D)` vs `(1, M, D)` pattern
This is the core broadcasting pattern for pairwise ops (IoU, distance, attention).  
Given `A` of shape `(N, D)` and `B` of shape `(M, D)`, compute `A[i] - B[j]` for all pairs — result shape `(N, M, D)`. No loops.

In [ ]:
A = np.array([[1,0,0],[0,1,0]], dtype=float)   # (2, 3)
B = np.array([[1,1,0],[0,0,1],[1,0,1]], dtype=float)  # (3, 3)

diff = None  # YOUR CODE — shape (2, 3, 3)
            # diff[i, j, :] = A[i] - B[j]

assert diff.shape == (2, 3, 3), diff.shape
assert np.allclose(diff[0, 0], A[0] - B[0]), diff[0, 0]  # [1,0,0]-[1,1,0]=[0,-1,0]
assert np.allclose(diff[1, 2], A[1] - B[2]), diff[1, 2]  # [0,1,0]-[1,0,1]=[-1,1,-1]

---
## Section D — Reductions & Sorting

### D1
Given `X` of shape `(N, C)` (logits), return the top-3 column indices per row sorted by **descending** value — shape `(N, 3)`. No loops.

In [ ]:
X = np.array([[0.1, 0.9, 0.3, 0.5, 0.2],
              [0.8, 0.1, 0.7, 0.2, 0.9]], dtype=float)

top3 = None  # YOUR CODE — shape (2, 3)

assert top3.shape == (2, 3)
# Row 0: sorted desc -> col 1(0.9), col 3(0.5), col 2(0.3)
assert list(top3[0]) == [1, 3, 2], top3[0]
# Row 1: col 4(0.9), col 0(0.8), col 2(0.7)
assert list(top3[1]) == [4, 0, 2], top3[1]

### D2
Given a 1D array `x`, compute the **rank** of each element (1 = smallest). No loops.  
Hint: `argsort` of `argsort` gives rank.

In [ ]:
x = np.array([40, 10, 30, 20, 50], dtype=float)

ranks = None  # YOUR CODE — shape (5,), 1-based

assert ranks.shape == (5,)
assert np.allclose(ranks, [4, 1, 3, 2, 5]), ranks

### D3
Given a 1D float array `x` and sorted bin edges `edges`, assign each element to a bin index `[0, num_bins)` where bin `i` contains `edges[i] <= x < edges[i+1]`. No loops.

In [ ]:
x     = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
edges = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])

bins = None  # YOUR CODE — shape (5,), values in [0, 4]

assert bins.shape == (5,)
assert np.allclose(bins, [0, 1, 2, 3, 4]), bins

### D4
Given `counts` array of shape `(C,)`, find all classes where the count is **below** the 25th percentile of all counts. Return their indices.

In [ ]:
counts = np.array([120, 30, 80, 15, 200, 10, 95, 40])

rare_classes = None  # YOUR CODE — indices where count < 25th percentile

assert set(rare_classes) == {3, 5}, rare_classes  # counts 15 and 10

### D5 — `np.unique` patterns
Given integer array `labels` of shape `(N,)`, return:
1. Unique classes sorted
2. Count of each unique class
3. Index of **first occurrence** of each unique class

In [109]:
labels = np.array([2, 0, 1, 2, 0, 0, 1, 3, 2])

classes    = np.unique(labels)  # YOUR CODE — [0, 1, 2, 3]
class_cnt  =  np.unique(labels, return_counts=True)[1]  # YOUR CODE — [3, 2, 3, 1]
first_occ  = np.unique(labels, return_index=True)[1]  # YOUR CODE — [1, 2, 0, 7]

assert np.allclose(classes,   [0, 1, 2, 3]), classes
assert np.allclose(class_cnt, [3, 2, 3, 1]), class_cnt
assert np.allclose(first_occ, [1, 2, 0, 7]), first_occ

---
## Section E — `np.pad` and `np.cumsum`

### E1
Pad a `(5,)` array with 2 zeros on the left and 3 zeros on the right.

In [ ]:
x = np.array([1, 2, 3, 4, 5])

padded = None  # YOUR CODE

assert padded.shape == (10,)
assert np.allclose(padded, [0,0,1,2,3,4,5,0,0,0]), padded

### E2
Pad a `(4, 4)` image with 1 pixel of zeros on all sides → `(6, 6)`.

In [ ]:
img = np.ones((4, 4))

padded = None  # YOUR CODE

assert padded.shape == (6, 6)
assert padded[0, 0] == 0 and padded[1, 1] == 1 and padded[-1, -1] == 0

### E3
Pad a `(N, H, W)` batch of images with 2 pixels on every spatial side but **do not pad the batch dimension** → `(N, H+4, W+4)`.

In [ ]:
imgs = np.random.randn(8, 6, 6)

padded = None  # YOUR CODE

assert padded.shape == (8, 10, 10)
assert np.allclose(padded[:, 2:-2, 2:-2], imgs), 'original content must be preserved'

### E4
Given `x` of shape `(N,)`, compute the **rolling sum** over a window of size `w=4` using `cumsum` only — no loops, no `np.convolve`. Output shape: `(N - w + 1,)`.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
w = 4

rolling = None  # YOUR CODE — shape (5,)

assert rolling.shape == (5,)
assert np.allclose(rolling, [10, 14, 18, 22, 26]), rolling
# 1+2+3+4=10, 2+3+4+5=14, ...

### E5
Given a binary array `x` of 0s and 1s, find the **starting indices** of all runs of consecutive 1s. Use `np.diff` — no loops.

In [ ]:
x = np.array([0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0])

run_starts = None  # YOUR CODE — [2, 7, 10]

assert np.allclose(run_starts, [2, 7, 10]), run_starts

---
## Section F — Coordinate & Grid Math

### F1
For a grid of shape `(H=5, W=5)`, use `np.meshgrid` to create 2D arrays `xx` and `yy` where `xx[i,j] = j` (column index) and `yy[i,j] = i` (row index).

In [ ]:
H, W = 5, 5

xx = None  # YOUR CODE — shape (5, 5)
yy = None  # YOUR CODE — shape (5, 5)

assert xx.shape == yy.shape == (5, 5)
assert xx[0, 3] == 3 and xx[4, 0] == 0, f'xx: {xx}'
assert yy[3, 0] == 3 and yy[0, 4] == 0, f'yy: {yy}'

### F2
Using your `xx` and `yy`, compute the Euclidean distance from center `(H//2, W//2)` for every pixel — shape `(H, W)`.

In [ ]:
cy, cx = H // 2, W // 2

dist = None  # YOUR CODE — shape (5, 5)

assert dist.shape == (5, 5)
assert dist[cy, cx] == 0.0, 'center should be 0'
assert np.isclose(dist[0, 0], np.sqrt(8)), dist[0, 0]  # distance from (2,2) to (0,0)

### F3 — flat index conversion
Given 2D indices `(rows, cols)` and array width `W`, compute the **flat (1D) index** `flat = row * W + col`. Then reverse: given flat indices and `W`, recover `(rows, cols)`.

In [ ]:
W    = 10
rows = np.array([0, 1, 3, 2])
cols = np.array([5, 0, 7, 9])

flat       = None  # YOUR CODE — [5, 10, 37, 29]
rows_back  = None  # YOUR CODE — recover rows from flat
cols_back  = None  # YOUR CODE — recover cols from flat

assert np.allclose(flat, [5, 10, 37, 29]), flat
assert np.allclose(rows_back, rows), rows_back
assert np.allclose(cols_back, cols), cols_back